In [ ]:
import sys
import os
sys.path.append('/media/ubuntu/sda/mouse_test/script/end2end')

import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface
from probeinterface import Probe, ProbeGroup, read_probeinterface

import numpy as np
from spikeinterface.core import concatenate_recordings
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from pathlib import Path
import pickle

from spikeinterface.core import get_template_extremum_channel
import scipy.spatial.distance
from scipy.sparse.csgraph import connected_components

print("="*70)
print("测试sorting重复性：读取phy_folder，生成detect_array，然后与GT匹配")
print("="*70)

# 输出目录
output_base = '/media/ubuntu/sda/mouse_test/processed_results/test'
os.makedirs(output_base, exist_ok=True)

# phy_folder路径
phy_folder = '/media/ubuntu/sda/mouse_test/processed_results/test/phy_folder'

session_name = 'mouse6_021322_natural_image_001'
print(f"\nSession: {session_name}")
print(f"Phy folder: {phy_folder}")

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


测试sorting重复性：使用第一个月数据进行sorting，然后与GT匹配

步骤1: 加载第一个月的数据并进行sorting

加载session: mouse6_021322_natural_image_001


In [ ]:
recording_raw = se.read_blackrock(file_path=f'/media/ubuntu/sda/data/mouse6/ns4/natural_image/{session_name}')
recording_recorded = recording_raw.remove_channels(["98", '31', '32']).time_slice(start_time=0, end_time=1500)

probe_30channel = read_probeinterface('/media/ubuntu/sda/data/probe.json')
recording_recorded = recording_recorded.set_probegroup(probe_30channel)

# 预处理（完全按照sorting.ipynb）
recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
recording_notch = spre.notch_filter(recording_f, freq=60)
recording_cmr = spre.common_reference(recording_notch, reference="global", operator="median")
recording_cmr = recording_cmr.rename_channels(['A-000', 'A-001', 'A-002', 'A-003', 'A-004',
                               'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
                               'A-0010', 'A-011', 'A-012', 'A-013', 'A-014',
                               'A-015', 'A-016', 'A-017', 'A-018', 'A-019',
                               'A-020', 'A-021', 'A-022', 'A-023', 'A-024',
                               'A-025', 'A-026', 'A-027', 'A-028', 'A-029'])

print(f"预处理完成: {recording_cmr.get_num_samples()} samples, {recording_cmr.get_num_channels()} channels")
print(f"Duration: {recording_cmr.get_num_samples() / recording_cmr.get_sampling_frequency():.2f} seconds")
# 保存为binary格式（Mountainsort4需要）
print("\n保存为binary格式...")
recording_preprocessed = recording_cmr.save(format="binary", n_jobs=20)
print("保存完成")

# Mountainsort4参数（完全按照sorting.ipynb）
default_params = {
    'detect_sign': -1,
    'adjacency_radius': 120,
    'freq_min': 300,
    'freq_max': 3000,
    'filter': True,
    'whiten': True,
    'num_workers': 20,
    'clip_size': 50,
    'detect_threshold': 5,
    'detect_interval': 3,
}

# 运行Mountainsort4
print("\n运行Mountainsort4 sorting...")
sorting_output_folder = os.path.join(output_base, 'mountainsort4_output')
firings_path = os.path.join(sorting_output_folder, 'sorter_output', 'firings.npz')

if os.path.exists(firings_path):
    print(f"发现已存在的sorting结果，直接加载: {firings_path}")
    sorting_mountainsort = se.NpzSortingExtractor(firings_path)
else:
    sorting_mountainsort = ss.run_sorter(
        sorter_name='mountainsort4',
        recording=recording_preprocessed,
        remove_existing_folder=True,
        folder=sorting_output_folder,
        **default_params
    )

print(f"Sorting完成，检测到 {len(sorting_mountainsort.unit_ids)} 个units")

# ===== 步骤2: 导出到phy格式 =====
print("\n" + "="*70)
print("步骤2: 导出到phy格式")
print("="*70)

phy_folder = os.path.join(output_base, 'phy_folder')
print(f"\n导出到phy格式: {phy_folder}")

analyzer_mountainsort = si.create_sorting_analyzer(
    sorting=sorting_mountainsort,
    recording=recording_preprocessed,
    format='binary_folder',
    folder=os.path.join(output_base, 'analyzer_binary')
)

# 计算必要的extensions
extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "noise_levels",
    "templates",
    "unit_locations",
]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
}

print("计算extensions...")
analyzer_mountainsort.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20)
print("Extensions计算完成")

# 导出到phy
import spikeinterface.exporters as sexp
sexp.export_to_phy(analyzer_mountainsort, phy_folder, verbose=True, n_jobs=20)
print("Phy导出完成")


预处理完成: 15000000 samples, 30 channels
Duration: 1500.00 seconds

保存为binary格式...
Use cache_folder=/tmp/spikeinterface_cache/tmp2tmyxz9c/15RZM7RV
write_binary_recording 
engine=process - n_jobs=20 - samples_per_chunk=10,000 - chunk_memory=585.94 KiB - total_memory=11.44 MiB - chunk_duration=1.00s


write_binary_recording (workers: 20 processes): 100%|██████████| 1500/1500 [00:02<00:00, 706.49it/s]


保存完成

运行Mountainsort4 sorting...
Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Sorting完成，检测到 44 个units

步骤2: 导出到phy格式

导出到phy格式: /media/ubuntu/sda/mouse_test/processed_results/test/phy_folder


estimate_sparsity (no parallelization): 100%|██████████| 1500/1500 [00:00<00:00, 59974.03it/s]

计算extensions...



/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/basesorting.py:380: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
noise_level (workers: 20 processes): 100%|██████████| 20/20 [00:00<00:00, 481.66it/s]

Extensions计算完成



extract PCs (workers: 20 processes): 100%|██████████| 1500/1500 [00:05<00:00, 267.64it/s]

Run:
phy template-gui  /media/ubuntu/sda/mouse_test/processed_results/test/phy_folder/params.py
Phy导出完成


In [16]:
# ===== 步骤2: 读取phy结果并进行post_sort（参考recordings_30channels_12_month.ipynb） =====
print("\n" + "="*70)
print("步骤2: 读取phy结果并进行post_sort")
print("="*70)

sampling_frequency = recording_cmr.get_sampling_frequency()

# 读取phy结果（注意：不排除noise cluster）
print(f"\n读取phy结果（不排除noise cluster）...")
sorting_curated_phy = se.read_phy(phy_folder)  # 不设置exclude_cluster_groups
print(f"读取到 {len(sorting_curated_phy.unit_ids)} 个units\n")
combined_output_base = '/media/ubuntu/sda/mouse_test/processed_results/test'
# 创建analyzer
print("创建analyzer并计算extensions...")
analyzer_curated_phy = si.create_sorting_analyzer(
    sorting=sorting_curated_phy, 
    recording=recording_cmr,  # 使用common reference后的recording
    format='binary_folder',
    folder=output_base + '/analyzer_curated_temp',
    n_jobs=20, 
    verbose=False
)

extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "templates",
    "unit_locations",
    "template_similarity"
]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
    "template_similarity": {"method": "cosine_similarity"}
}

analyzer_curated_phy.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
print("完成extensions计算\n")

# 获取neuron信息（整个recording）
templates_ext = analyzer_curated_phy.get_extension("templates")
templates_dense = templates_ext.data["average"]
sparsity = analyzer_curated_phy.sparsity
unit_locations_ext = analyzer_curated_phy.get_extension("unit_locations")
unit_locations = unit_locations_ext.get_data()
channel_locations = analyzer_curated_phy.get_channel_locations()

# 处理merge逻辑
if unit_locations.shape[1] >= 2:
    unit_distances = scipy.spatial.distance.cdist(
        unit_locations[:, :2], 
        unit_locations[:, :2], 
        metric="euclidean"
    )
else:
    unit_distances = scipy.spatial.distance.cdist(
        unit_locations, 
        unit_locations, 
        metric="euclidean"
    )

template_similarity_ext = analyzer_curated_phy.get_extension("template_similarity")
template_similarity = template_similarity_ext.get_data()

distance_threshold = 10.0
similarity_threshold = 0.95
num_units = len(analyzer_curated_phy.unit_ids)
pair_mask = np.zeros((num_units, num_units), dtype=bool)

for i in range(num_units):
    for j in range(i + 1, num_units):
        if unit_distances[i, j] < distance_threshold and template_similarity[i, j] > similarity_threshold:
            pair_mask[i, j] = True
            pair_mask[j, i] = True

n_components, labels = connected_components(
    csgraph=pair_mask, 
    directed=False, 
    return_labels=True
)

merge_unit_groups = []
unit_ids_list = analyzer_curated_phy.unit_ids
for component_id in range(n_components):
    unit_indices = np.where(labels == component_id)[0]
    if len(unit_indices) > 1:
        group = [unit_ids_list[i] for i in unit_indices]
        merge_unit_groups.append(group)

# 应用merge（如果有需要merge的units）
if len(merge_unit_groups) > 0:
    print(f"发现 {len(merge_unit_groups)} 组需要merge的units，开始merge...")
    analyzer_merged = analyzer_curated_phy.merge_units(
        merge_unit_groups=merge_unit_groups,
        censor_ms=0.3,
        merging_mode="hard",
        new_id_strategy="append",
        format='binary_folder',
        folder=combined_output_base + '/analyzer_merged',
        verbose=True,
        n_jobs=20
    )
    
    analyzer_merged.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
    
    templates_ext_final = analyzer_merged.get_extension("templates")
    templates_dense_final = templates_ext_final.data["average"]
    sparsity_final = analyzer_merged.sparsity
    unit_locations_ext_final = analyzer_merged.get_extension("unit_locations")
    unit_locations_final = unit_locations_ext_final.get_data()
    channel_locations_final = analyzer_merged.get_channel_locations()
    sorting_final = analyzer_merged.sorting
    unit_ids_list_final = analyzer_merged.unit_ids
    
    # 生成position_waveforms
    position_waveforms_final = []
    for unit_id in unit_ids_list_final:
        unit_index = analyzer_merged.sorting.id_to_index(unit_id)
        template_dense_unit = templates_dense_final[unit_index, :, :]
        template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
        sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
        
        if len(sparse_channel_indices) == 0:
            position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
            position_waveforms_final.append(position_waveform)
            continue
        
        sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
        unit_location = unit_locations_final[unit_index, :2]
        
        distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
        epsilon = 1e-10
        weights = 1.0 / (distances + epsilon)
        weights = weights / np.sum(weights)
        
        position_waveform = np.dot(template_sparse_unit, weights)
        position_waveforms_final.append(position_waveform)
    
    position_waveforms_final = np.array(position_waveforms_final)
    extremum_channels_final = get_template_extremum_channel(
        analyzer_merged, 
        peak_sign="neg",
        outputs="id"
    )
    
    channel_ids_list = list(analyzer_merged.recording.get_channel_ids())
else:
    print("无需merge units\n")
    # 不需要merge，使用原始结果
    templates_ext_final = analyzer_curated_phy.get_extension("templates")
    templates_dense_final = templates_ext_final.data["average"]
    sparsity_final = analyzer_curated_phy.sparsity
    unit_locations_ext_final = analyzer_curated_phy.get_extension("unit_locations")
    unit_locations_final = unit_locations_ext_final.get_data()
    channel_locations_final = analyzer_curated_phy.get_channel_locations()
    sorting_final = analyzer_curated_phy.sorting
    unit_ids_list_final = unit_ids_list
    
    # 生成position_waveforms
    position_waveforms_final = []
    for unit_id in unit_ids_list_final:
        unit_index = analyzer_curated_phy.sorting.id_to_index(unit_id)
        template_dense_unit = templates_dense_final[unit_index, :, :]
        template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
        sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
        
        if len(sparse_channel_indices) == 0:
            position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
            position_waveforms_final.append(position_waveform)
            continue
        
        sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
        unit_location = unit_locations_final[unit_index, :2]
        
        distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
        epsilon = 1e-10
        weights = 1.0 / (distances + epsilon)
        weights = weights / np.sum(weights)
        
        position_waveform = np.dot(template_sparse_unit, weights)
        position_waveforms_final.append(position_waveform)
    
    position_waveforms_final = np.array(position_waveforms_final)
    extremum_channels_final = get_template_extremum_channel(
        analyzer_curated_phy, 
        peak_sign="neg",
        outputs="id"
    )
    
    channel_ids_list = list(analyzer_curated_phy.recording.get_channel_ids())

# 计算每个unit的channel_id（template中值不为0的通道）
# recording的channel_ids已经是probe的contact_ids（已在读取probe时设置）
print("计算每个unit的channel_id...")
channel_ids_dict = {}  # {unit_id: [contact_id1, contact_id2, ...]}
for idx, unit_id in enumerate(unit_ids_list_final):
    unit_index = sorting_final.id_to_index(unit_id)
    template_unit = templates_dense_final[unit_index, :, :]  # (n_samples, n_channels)
    
    # 找到template中值不为0的通道
    non_zero_channels = []
    for ch_idx in range(template_unit.shape[1]):  # 遍历channels（最后一个维度）
        if np.any(template_unit[:, ch_idx] != 0):  # 检查该通道在所有时间点的值
            # recording的channel_id已经是contact_id，直接使用
            contact_id = str(channel_ids_list[ch_idx])
            non_zero_channels.append(contact_id)
    
    channel_ids_dict[unit_id] = non_zero_channels

print(f"完成channel_id计算，共处理{len(channel_ids_dict)}个units\n")

# 计算channel_snr（每个unit的各个channel的SNR）
print("计算channel_snr...")
n_channels = recording_cmr.get_num_channels()

# 计算noise_std（使用前10秒的数据）
duration_samples = int(10 * sampling_frequency)  # 10秒
max_samples = min(duration_samples, recording_cmr.get_num_samples())
traces = recording_cmr.get_traces(start_frame=0, end_frame=max_samples)  # (n_timepoints, n_channels)

noise_std_detect = np.median(np.abs(traces) / 0.6745, axis=0)  # (n_channels,)

all_spike_times = []
all_spike_unit_ids = []
for unit_id in unit_ids_list_final:
    spike_train = sorting_final.get_unit_spike_train(unit_id)
    all_spike_times.extend(spike_train.tolist())
    all_spike_unit_ids.extend([unit_id] * len(spike_train))

n_spikes_total = len(all_spike_times)
n_spikes_sample = min(1000, n_spikes_total)
if n_spikes_sample > 0:
    random_indices = np.random.choice(n_spikes_total, size=n_spikes_sample, replace=False)
    sampled_spike_times = [all_spike_times[i] for i in random_indices]
    sampled_spike_unit_ids = [all_spike_unit_ids[i] for i in random_indices]
else:
    sampled_spike_times = []
    sampled_spike_unit_ids = []

# 提取这些spike的waveform并计算每个channel的负值amplitude
left_sample = 10
right_sample = 20
window_size = left_sample + right_sample

channel_snr_dict = {} 

for unit_id in unit_ids_list_final:
    channel_snr_dict[unit_id] = {}
    unit_spike_times = [st for st, uid in zip(sampled_spike_times, sampled_spike_unit_ids) if uid == unit_id]
    
    if len(unit_spike_times) == 0:
        unit_spike_times = sorting_final.get_unit_spike_train(unit_id).tolist()
        if len(unit_spike_times) > 1000:
            unit_spike_times = np.random.choice(unit_spike_times, size=1000, replace=False).tolist()
    
    unit_waveforms = []  # List of (n_channels, window_size)
    valid_spike_times = []
    
    for spike_time in unit_spike_times:
        start = spike_time - left_sample
        end = spike_time + right_sample

        # recording_cmr合并后只有一个segment，直接使用全局采样点索引
        # 确保索引在有效范围内
        if start < 0:
            start = 0
        if end > recording_cmr.get_num_samples():
            end = recording_cmr.get_num_samples()
        
        waveform = recording_cmr.get_traces(start_frame=start, end_frame=end)  # (n_timepoints, n_channels)
        unit_waveforms.append(waveform)
        valid_spike_times.append(spike_time)
    
    if len(unit_waveforms) == 0:
        continue
    
    unit_waveforms = np.array(unit_waveforms)  # (n_spikes, n_timepoints, n_channels)
    
    spike_time_values = unit_waveforms[:, left_sample, :]  # (n_spikes, n_channels) - 每个spike在spike_time时刻各个channel的值
    
    channel_amplitudes = np.mean(spike_time_values, axis=0)  # (n_channels,) - 每个channel的平均值（在spike_time时刻）
    channel_snr = np.abs(channel_amplitudes) / noise_std_detect  # (n_channels,)
    
    # 只保存 channel_ids_dict[unit_id] 中列出的通道的 SNR
    unit_channel_ids = channel_ids_dict.get(unit_id, [])  # 获取该unit的channel_id列表
    
    for ch_idx, snr_value in enumerate(channel_snr):
        channel_id = str(channel_ids_list[ch_idx])
        # 只保存 channel_ids_dict 中列出的通道
        if channel_id in unit_channel_ids:
            channel_snr_dict[unit_id][channel_id] = float(snr_value)

print(f"完成channel_snr计算，共处理{len(channel_snr_dict)}个units\n")

# 生成整体的neuron_inf（所有units）
neuron_inf_all = {}
for idx, unit_id in enumerate(unit_ids_list_final):
    neuron_inf_all[unit_id] = {
        'location_x': float(unit_locations_final[idx, 0]),
        'location_y': float(unit_locations_final[idx, 1]),
        'position_waveform': position_waveforms_final[idx],
        'extremum_channel': extremum_channels_final[unit_id],
        'channel_id': channel_ids_dict[unit_id],
        'channel_snr': channel_snr_dict.get(unit_id, {})  # 添加channel_snr字段
    }

# 生成detect_array（所有spikes）
print("生成detect_array...")
spike_vector_final = sorting_final.to_spike_vector()
detect_data_all = []

for spike in spike_vector_final:
    unit_index = spike['unit_index']
    unit_id = sorting_final.unit_ids[unit_index]
    sample_index = spike['sample_index']  # 采样点索引（单个session，从0开始）
    
    extremum_channel = extremum_channels_final[unit_id]
    
    detect_data_all.append({
        'time': sample_index,
        'unit_id': unit_id,
        'extremum_channel': str(extremum_channel),
    })

detect_array_df = pd.DataFrame(detect_data_all)
print(f"完成detect_array生成，共{len(detect_array_df)}个spikes\n")

# 保存结果
print("保存结果...")
with open(output_base + '/neuron_inf.pickle', 'wb') as f:
    pickle.dump(neuron_inf_all, f)
detect_array_df.to_csv(output_base + '/detect_array.csv', index=False)
print(f"已保存到: {output_base}/neuron_inf.pickle 和 {output_base}/detect_array.csv\n")

# ===== 步骤3: 与GT进行匹配 =====
print("\n" + "="*70)
print("步骤3: 与GT进行匹配")
print("="*70)

# 加载GT数据
gt_folder = '/media/ubuntu/sda/mouse_test/script/end2end/spike_sorting_model/clique_0'
gt_detect_array_path = os.path.join(gt_folder, session_name, 'gt_detect_array.csv')
gt_neuron_inf_path = os.path.join(gt_folder, session_name, 'neuron_inf.pickle')

if not os.path.exists(gt_detect_array_path) or not os.path.exists(gt_neuron_inf_path):
    raise FileNotFoundError(f"GT文件不存在: {gt_detect_array_path} 或 {gt_neuron_inf_path}")

gt_detect_array = pd.read_csv(gt_detect_array_path)
with open(gt_neuron_inf_path, 'rb') as f:
    gt_neuron_inf = pickle.load(f)

print(f"GT spikes数量: {len(gt_detect_array):,}")
print(f"GT neurons数量: {len(gt_neuron_inf):,}")

# 构建detect_array和gt_array（格式：[time, channel]）
print("\n构建detect_array和gt_array...")
channel_names = recording_cmr.get_channel_ids()
channel_name_to_idx = {name: idx for idx, name in enumerate(channel_names)}

# detect_array
detect_times = detect_array_df['time'].values
detect_channels = detect_array_df['extremum_channel'].values
detect_channel_indices = [channel_name_to_idx.get(str(ch), -1) for ch in detect_channels]
valid_detect_mask = np.array([idx >= 0 for idx in detect_channel_indices])
detect_array = np.column_stack([
    detect_times[valid_detect_mask],
    np.array(detect_channel_indices)[valid_detect_mask]
])

# gt_array
gt_times = gt_detect_array['time'].values
gt_channels = gt_detect_array['extremum_channel'].values
gt_channel_indices = [channel_name_to_idx.get(str(ch), -1) for ch in gt_channels]
valid_gt_mask = np.array([idx >= 0 for idx in gt_channel_indices])
gt_array = np.column_stack([
    gt_times[valid_gt_mask],
    np.array(gt_channel_indices)[valid_gt_mask]
])

gt_array = gt_array[gt_array[:, 0] < 15000000]
print(f"有效detect spikes: {len(detect_array):,}")
print(f"有效GT spikes: {len(gt_array):,}")

# GT匹配函数
def map_gt_annotation(detect_array, gt_array):
    gt_label_array1 = np.zeros((detect_array.shape[0],)) - 1
    
    for ind, i in enumerate(detect_array):
        f = 1
        indj = np.where(gt_array[:, 0] == i[0])[0]
        for j in indj:
            if gt_array[j, 1] == i[1]:
                f = 0
                break
        if f:
            indj = np.where(gt_array[:, 0] == i[0] - 1)[0]
            for j in indj:
                if gt_array[j, 1] == i[1]:
                    f = 0
                    break
        if f:
            indj = np.where(gt_array[:, 0] == i[0] + 1)[0]
            for j in indj:
                if gt_array[j, 1] == i[1]:
                    f = 0
                    break
        if f == 0:
            gt_label_array1[ind] = j
    
    return gt_label_array1

# 匹配
print("\n进行GT匹配...")
gt_label_array = map_gt_annotation(detect_array, gt_array)
matched_indices = np.where(gt_label_array > -1)[0]
n_matched = len(matched_indices)
n_detected = len(detect_array)
n_gt = len(gt_array)

print(f"\n匹配结果:")
print(f"  检测到的spikes: {n_detected:,}")
print(f"  GT spikes: {n_gt:,}")
print(f"  匹配的spikes: {n_matched:,}")
print(f"  召回率 (Recall): {n_matched/n_gt:.4f} ({n_matched/n_gt*100:.2f}%)")
print(f"  精确率 (Precision): {n_matched/n_detected:.4f} ({n_matched/n_detected*100:.2f}%)")

f1_score = 2 * (n_matched/n_gt) * (n_matched/n_detected) / ((n_matched/n_gt) + (n_matched/n_detected)) if (n_matched/n_gt + n_matched/n_detected) > 0 else 0
print(f"  F1 Score: {f1_score:.4f}")

n_noise = n_detected - n_matched
noise_ratio = n_noise / n_detected if n_detected > 0 else 0
print(f"  Noise比例: {noise_ratio:.4f} ({noise_ratio*100:.2f}%)")

# 分析未匹配的GT spikes
matched_gt_indices = gt_label_array[matched_indices].astype(int)
all_gt_indices = np.arange(len(gt_array))
unmatched_gt_indices = np.setdiff1d(all_gt_indices, matched_gt_indices)
n_unmatched_gt = len(unmatched_gt_indices)

print(f"\n未匹配的GT spikes: {n_unmatched_gt:,} ({n_unmatched_gt/n_gt*100:.2f}%)")

if n_unmatched_gt > 0:
    unmatched_gt_df = gt_detect_array.iloc[valid_gt_mask][unmatched_gt_indices]
    unmatched_neuron_ids = unmatched_gt_df['unit_id'].values
    
    unique_neurons, neuron_counts = np.unique(unmatched_neuron_ids, return_counts=True)
    neuron_counts_sorted_idx = np.argsort(neuron_counts)[::-1]
    
    print(f"\n未匹配GT spikes的neuron分布（前10个）:")
    for i, idx in enumerate(neuron_counts_sorted_idx[:10]):
        neuron_id = unique_neurons[idx]
        count = neuron_counts[idx]
        percentage = count / n_unmatched_gt * 100
        print(f"  Neuron {neuron_id}: {count:,} spikes ({percentage:.2f}%)")

print("\n" + "="*70)
print("测试完成！结果已保存到:")
print(f"  {output_base}")
print("="*70)


步骤2: 读取phy结果并进行post_sort

读取phy结果（不排除noise cluster）...
读取到 44 个units

创建analyzer并计算extensions...


compute_waveforms (workers: 20 processes): 100%|██████████| 1500/1500 [00:11<00:00, 128.63it/s]


完成extensions计算

发现 1 组需要merge的units，开始merge...
compute_waveforms 
engine=process - n_jobs=20 - samples_per_chunk=10,000 - chunk_memory=585.94 KiB - total_memory=11.44 MiB - chunk_duration=1.00s


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/sortinganalyzer.py:2656: DeprecationWarning: The method 'cosine_similarity' is deprecated and will be removed in the next version. Use 'cosine' instead.
  params = self._set_params(**params)
compute_waveforms (workers: 20 processes): 100%|██████████| 1500/1500 [00:13<00:00, 111.37it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/sortinganalyzer.py:2656: DeprecationWarning: The method 'cosine_similarity' is deprecated and will be removed in the next version. Use 'cosine' instead.
  params = self._set_params(**params)


计算每个unit的channel_id...
完成channel_id计算，共处理43个units

计算channel_snr...
完成channel_snr计算，共处理43个units

生成detect_array...
完成detect_array生成，共362015个spikes

保存结果...
已保存到: /media/ubuntu/sda/mouse_test/processed_results/test/neuron_inf.pickle 和 /media/ubuntu/sda/mouse_test/processed_results/test/detect_array.csv


步骤3: 与GT进行匹配
GT spikes数量: 856,715
GT neurons数量: 31

构建detect_array和gt_array...
有效detect spikes: 362,015
有效GT spikes: 325,084

进行GT匹配...

匹配结果:
  检测到的spikes: 362,015
  GT spikes: 325,084
  匹配的spikes: 266,813
  召回率 (Recall): 0.8208 (82.08%)
  精确率 (Precision): 0.7370 (73.70%)
  F1 Score: 0.7766
  Noise比例: 0.2630 (26.30%)

未匹配的GT spikes: 58,271 (17.92%)


KeyError: "None of [Index([     2,      3,      4,      5,      6,      7,      8,     12,     14,\n           18,\n       ...\n       325026, 325029, 325033, 325041, 325046, 325048, 325055, 325059, 325066,\n       325083],\n      dtype='int64', length=58271)] are in the [columns]"